<a href="https://colab.research.google.com/github/Jsanch1020/is201/blob/main/Ex07-Ken_French.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Investments: Theory and Data Analysis**, Bates, Boyer, and Fletcher

# Examples Chapter 7: The Ken French Data Library
In this example we illustrate how to access data in the Ken French Data Library using a Python function that wraps the library’s HTTP endpoint in an API-style interface.

### Imports and Setup

In [12]:
# Import packages
%pip install -q simple-finance
import simple_finance as sf
import requests
import pandas as pd
import numpy as np

# Establish display options when printing DataFrames
pd.set_option('display.max_columns', None)   # Show all columns without truncation
pd.set_option('display.width', 1000)   # Set the display width so output stays on one line
pd.set_option("display.max_rows", 20) # Force truncation if DataFrame has more than 20 rows

### Load in Data Using the Python Function

####Function: `get_ff_strategies()`

**Inputs**  
* `stype`: type of sstrategy. This input can be one of five strings:  
  - [`beta`](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_port_form_BETA.html)  
  - [`momentum`](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_mom_factor.html)  
  - [`shortermreversal`](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_st_rev_factor.html)
  - [`accruals`](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_port_form_AC.html)  
  - [`value`](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_form_btm.html)  
By clicking on each link you can learn more about each sorting strategy.
* `start_date` (*optional*): the starting date for the data entered as a string: `yyyy-mm`. If not included the data will begin on the earliest possible date.
* `end_date` (*optional*): the ending date for the data entered as a string `yyyy-mm`.  If not included the data will end on the latest possible date.
* `details` (*optional*): If `True`, prints descriptive information about the trading strategy to the terminal. Default is `False`.
* `factors` (*optional*): If equal to the string `FF5` will include all factors from the Fama-French Five factor model, and the return on short-term t-bills. Otherwise, default is to include all factors from the Fama-French Three factor model, and the return on short-term t-bills.


**Output**  
DataFrame
* Decile portfolios of the chosen strategy.
* Factor returns
* Return on short-term T-bills.

** Example **  
`df=get_ff_strategies('momentum',start_date='2020-01', details=True)`  
This call returns decile portfolio returns for the momentum strategy beginning in January 2020 and continuing to the present, and prints descriptive information about the strategy to the terminal.


In [13]:
# Get returns on short-term reveral deciles
df=sf.get_ff_strategies('momentum', details=False)


In [14]:
print(df)


          Dec 1   Dec 2   Dec 3   Dec 4   Dec 5   Dec 6   Dec 7   Dec 8   Dec 9  Dec 10  mkt-rf     smb     hml      rf
date                                                                                                                   
1927-01 -0.0332 -0.0454  0.0267 -0.0029 -0.0041  0.0093  0.0030  0.0071 -0.0014 -0.0024 -0.0005 -0.0032  0.0458  0.0025
1927-02  0.0739  0.0601  0.0703  0.0746  0.0434  0.0398  0.0299  0.0320  0.0414  0.0704  0.0417  0.0007  0.0272  0.0026
1927-03 -0.0323 -0.0305 -0.0384 -0.0480 -0.0046 -0.0235  0.0196  0.0049  0.0035  0.0613  0.0014 -0.0177 -0.0238  0.0030
1927-04  0.0128 -0.0301 -0.0244  0.0043  0.0119 -0.0016  0.0203 -0.0074  0.0181  0.0549  0.0047  0.0039  0.0065  0.0025
1927-05  0.0271  0.0454  0.0597  0.0314  0.0614  0.0593  0.0522  0.0632  0.0819  0.0592  0.0545  0.0155  0.0480  0.0030
...         ...     ...     ...     ...     ...     ...     ...     ...     ...     ...     ...     ...     ...     ...
2025-10  0.0028 -0.0025  0.0430  0.0077 



```
# This is formatted as code
```

### Define Tangent Portfolio


In [15]:
df = df.copy()  # avoid SettingWithCopyWarning

# Define the zero-cost portfolio
df['Dec10_Dec1'] = df['Dec 10'] - df['Dec 1']

# Estimate Expected excess returns
expected_returns = df[['mkt-rf', 'Dec10_Dec1']].mean().to_numpy()

# Estimate the covariance matrix
covariance_matrix = df[['mkt-rf', 'Dec10_Dec1']].cov().to_numpy()
print(expected_returns)
print(covariance_matrix)

[0.00687193 0.01126454]
[[ 0.00282407 -0.00153151]
 [-0.00153151  0.0062457 ]]


In [16]:
tangent_weights, tangent_return, tangent_volatility = sf.tangent_portfolio(expected_returns, covariance_matrix, factors=True)

print(tangent_weights)
print(tangent_return)
print(tangent_volatility)

[1.         0.70349636]
0.014796494100263697
0.06132123872784755


### Test Trading Strategy

In [17]:
# For comparison
#tangent_weights = np.array([1, 0.250])

# Create realized excess portfolio returns
excess = df[['mkt-rf', 'Dec10_Dec1']].to_numpy()   # shape (T,2)

# Calculate the excess returns as a 1D array
# @ performs an inner product
excess_array = excess @ tangent_weights

# Convert the Series to a DataFrame
excess = pd.DataFrame(excess_array, index=df.index, columns=['excess'])

# compute total return
total_return = excess['excess'] + df['rf']

# Get monthly mean returns, volatility, and sharpe
# Call .mean() to get a scalar mean from the total_return Series
mean_monthly = total_return.mean()
# Select the 'excess' column before calling .mean() as 'excess' is a DataFrame
mean_excess_monthly  = excess['excess'].mean()
# Select the 'excess' column before calling .std() as 'excess' is a DataFrame
std_monthly  = total_return.std()
sharpe_monthly = mean_excess_monthly / std_monthly

# annualize
ann_return = 12 * mean_monthly
ann_vol    = np.sqrt(12) * std_monthly
ann_sharpe = sharpe_monthly * np.sqrt(12)

# Print results
print(f"Average Return: {ann_return:.3f}")
print(f"Annualized Volatility: {ann_vol:.3f}")
print(f"Annualized Sharpe Ratio: {ann_sharpe:.3f}")


Average Return: 0.210
Annualized Volatility: 0.213
Annualized Sharpe Ratio: 0.835
